# Задание 4 — Подготовка на данните и първоначално извличане на признаци

**CRISP-DM фаза:** Data Preparation  
**Предмет:** Невронни мрежи - практика

---

## 1. Въведение и Цели

Целта на този ноутбук е да трансформира "суровите" данни (в нашия случай, обединените V9 данни) във формат, подходящ за машинно обучение. Това включва почистване, създаване на нови признаци (Feature Engineering) и скалиране.

**Основни стъпки:**
1.  **Load & Merge**: Зареждане на съществуващите V9 данни и обединяването им в един `raw_df`.
2.  **Missing Values**: Стратегия за липсващи данни (ако има такива).
3.  **Outliers**: Анализ и третиране на екстремни стойности.
4.  **Feature Engineering**: Създаване на нови променливи на базата на домейн знание.
5.  **Data Splitting**: Разделяне на Train/Val/Test (Stratified).
6.  **Pipeline**: Създаване на `scikit-learn` Pipeline за автоматична обработка.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin

# Настройки за визуализация
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 2. Зареждане на Данните (Data Loading)

Ще използваме `processed_v9` като източник, но ще го обединим, за да симулираме "суров" входен поток, върху който тепърва ще прилагаме трансформации.

In [ ]:
DATA_DIR = '../data/processed_v9'

def load_and_merge_data(data_dir):
    try:
        X_train = pd.read_csv(os.path.join(data_dir, 'X_train.csv'))
        y_train = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))
        X_val = pd.read_csv(os.path.join(data_dir, 'X_val.csv'))
        y_val = pd.read_csv(os.path.join(data_dir, 'y_val.csv'))
        X_test = pd.read_csv(os.path.join(data_dir, 'X_test.csv'))
        y_test = pd.read_csv(os.path.join(data_dir, 'y_test.csv'))
        
        # Обединяване
        df_train = pd.concat([X_train, y_train], axis=1)
        df_val = pd.concat([X_val, y_val], axis=1)
        df_test = pd.concat([X_test, y_test], axis=1)
        
        raw_df = pd.concat([df_train, df_val, df_test], axis=0).reset_index(drop=True)
        return raw_df
    except FileNotFoundError as e:
        print(f"Грешка при зареждане: {e}")
        return None

raw_df = load_and_merge_data(DATA_DIR)

if raw_df is not None:
    print(f"Успешно заредени {raw_df.shape[0]} записа с {raw_df.shape[1]} колони.")
    display(raw_df.head())

## 3. Стратегия за Липсващи Стойности (Missing Values Strategy)

Въпреки че нашият синтетичен набор е "чист", в реална среда винаги трябва да имаме стратегия. Тук ще дефинираме такава и ще я включим в Pipeline-а.

**Стратегия:**
*   **Числови данни (напр. `Age`, `Monthly_Salary`):** Imputation with **Median** (по-устойчиво на outliers от Mean).
*   **Категорийни данни (напр. `Department`):** Imputation with **Constant ('Missing')** или **Mode**.

In [ ]:
# Проверка за липсващи стойности
missing_values = raw_df.isnull().sum()
print("Липсващи стойности по колони:")
print(missing_values[missing_values > 0])

# Демонстрация: Дори да е празно сега, Pipeline-ът ще го обработи автоматично.

## 4. Обработка на Отклонения (Outlier Handling)

Разглеждаме `Monthly_Salary` и `Overtime_Hours` за екстремни стойности, които могат да изкривят скалирането (напр. MinMax).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(x=raw_df['Monthly_Salary'], ax=axes[0], color='skyblue')
axes[0].set_title('Boxplot: Monthly Salary')

sns.boxplot(x=raw_df['Overtime_Hours'], ax=axes[1], color='salmon')
axes[1].set_title('Boxplot: Overtime Hours')

plt.show()

**Решение за Outliers:**
Ще приложим **Capping (Winsorization)** или **Robust Scaler** в Pipeline-а. За целите на заданието, ще дефинираме къстъм трансформър за capping.

In [ ]:
class OutlierCapper(BaseEstimator, TransformerMixin):
    def __init__(self, factor=1.5):
        self.factor = factor
        self.caps_ = {}

    def fit(self, X, y=None):
        # X is expected to be a DataFrame
        for col in X.columns:
            if pd.api.types.is_numeric_dtype(X[col]):
                q1 = X[col].quantile(0.25)
                q3 = X[col].quantile(0.75)
                iqr = q3 - q1
                upper_cap = q3 + (self.factor * iqr)
                lower_cap = q1 - (self.factor * iqr)
                self.caps_[col] = (lower_cap, upper_cap)
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col, (lower, upper) in self.caps_.items():
            X_copy[col] = X_copy[col].clip(lower=lower, upper=upper)
        return X_copy

## 5. Feature Engineering (Създаване на нови признаци)

Ще създадем няколко нови променливи, които (според EDA) имат логическа връзка с напускането:
1.  **Burnout_Index**: Комбинация от `Overtime` и `Projects`.
2.  **Salary_Per_Year**: Заплата разделена на годините стаж (показва растеж на дохода).
3.  **Efficiency**: `Performance_Score` / `Work_Hours` (приблизителна ефективност).

In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        
        # 1. Burnout Index (Interaction)
        # Нормализираме грубо преди умножение за да са в един порядък
        X['Burnout_Index'] = (X['Overtime_Hours'] * X['Projects_Handled'])
        
        # 2. Salary Growth Potential
        # Избягваме делене на нула (Years_At_Company + 1)
        X['Salary_Per_Year'] = X['Monthly_Salary'] / (X['Years_At_Company'] + 1)
        
        # 3. Efficiency Metric
        X['Efficiency'] = X['Performance_Score'] / X['Work_Hours_Per_Week']
        
        return X

## 6. Разделяне на Данните (Data Splitting)

Преди да пуснем Pipeline-а (който включва fit), трябва да разделим данните, за да избегнем **Data Leakage** (изтичане на информация от тестовия сет).

Ще използваме **Stratified Split**, за да запазим процента на напусналите (Resigned) във всички множества.

In [ ]:
# Дефиниране на Features (X) и Target (y)
# Note: Изключваме 'Employee_ID' (идентификатор) и 'Employee_Satisfaction_Score' (target leakage risk от Assignment 3)
drop_cols = ['Resigned', 'Employee_ID', 'Employee_Satisfaction_Score', 'Hire_Date']

X = raw_df.drop(columns=drop_cols, errors='ignore')
y = raw_df['Resigned']

# 1. Train / Temp Split (80% Train, 20% Temp)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 2. Temp -> Val / Test Split (50/50 от останалите 20% -> 10% Val, 10% Test)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train size: {X_train.shape}, Class 1 ratio: {y_train.mean():.2%}")
print(f"Val size:   {X_val.shape},   Class 1 ratio: {y_val.mean():.2%}")
print(f"Test size:  {X_test.shape},  Class 1 ratio: {y_test.mean():.2%}")

## 7. Изграждане на Pipeline

Ще създадем `ColumnTransformer`, който прилага различни обработки към различни типове колони:

1.  **Numeric Features**: Impute -> Outlier Cap -> Engineer -> Scale (StandardScaler)
2.  **Categorical Features**: Impute -> Encode (OneHot)

*Забележка: Scikit-learn Pipelines са последователни. За да включим къстъм Feature Engineering, който променя колоните, понякога е по-лесно да го направим глобално или като първа стъпка, но тук ще го интегрираме.*

In [ ]:
# Дефиниране на колоните
numeric_features = ['Age', 'Years_At_Company', 'Monthly_Salary', 'Work_Hours_Per_Week', 
                    'Projects_Handled', 'Overtime_Hours', 'Sick_Days', 
                    'Remote_Work_Frequency', 'Team_Size', 'Training_Hours', 'Promotions']

categorical_features = ['Department', 'Gender', 'Job_Title', 'Education_Level']

# Пайплайн за числови данни
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('capper', OutlierCapper(factor=2.0)), # По-висок фактор за по-малко агресивно рязане
    ('scaler', StandardScaler())
])

# Пайплайн за категорийни данни
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Главен Feature Engineering (преди разделянето по типове, защото ползва комбинации)
# За улеснение в sklearn, често правим FE върху целия DF или правим къстъм препроцесор.
# Тук ще приложим FeatureEngineer ръчно преди ColumnTransformer за прозрачност.

fe = FeatureEngineer()
X_train_fe = fe.transform(X_train)
X_val_fe = fe.transform(X_val)
X_test_fe = fe.transform(X_test)

# Обновяваме списъка с числови колони, защото добавихме нови
numeric_features_extended = numeric_features + ['Burnout_Index', 'Salary_Per_Year', 'Efficiency']

# Сглобяване на ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features_extended),
        ('cat', categorical_transformer, categorical_features)
    ])

# Fit & Transform
# ВАЖНО: Fit само върху Train!
X_train_processed = preprocessor.fit_transform(X_train_fe)
X_val_processed = preprocessor.transform(X_val_fe)
X_test_processed = preprocessor.transform(X_test_fe)

print("Preprocessing complete.")
print(f"X_train_processed shape: {X_train_processed.shape}")

## 8. Проверка и Валидиране (Feature Sanity Checks)

Нека визуализираме новите признаци след скалирането, за да сме сигурни, че разпределенията изглеждат нормално (около 0).

In [ ]:
# Възстановяване на имена на колони за визуализация (OneHot ги променя)
ohe_feature_names = preprocessor.named_transformers_['cat']['onehot'].get_feature_names_out(categorical_features)
all_feature_names = numeric_features_extended + list(ohe_feature_names)

X_train_df = pd.DataFrame(X_train_processed, columns=all_feature_names)

plt.figure(figsize=(12, 6))
sns.kdeplot(data=X_train_df[['Monthly_Salary', 'Burnout_Index', 'Efficiency']], fill=True)
plt.title('Разпределение на скалирани признаци (StandardScaler)')
plt.xlabel('Z-Score')
plt.show()

## 9. Запазване на Готовите Данни

Записваме обработените данни в нова директория `data/prepared`, готови за моделиране.

In [ ]:
OUTPUT_DIR = '../data/prepared'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Запазване като CSV (за лесна четимост) или Parquet (за ефективност)
pd.DataFrame(X_train_processed, columns=all_feature_names).to_csv(f'{OUTPUT_DIR}/X_train_prep.csv', index=False)
pd.DataFrame(X_val_processed, columns=all_feature_names).to_csv(f'{OUTPUT_DIR}/X_val_prep.csv', index=False)
pd.DataFrame(X_test_processed, columns=all_feature_names).to_csv(f'{OUTPUT_DIR}/X_test_prep.csv', index=False)

# Target-ите не са променени, но ги запазваме за удобство
y_train.to_csv(f'{OUTPUT_DIR}/y_train_prep.csv', index=False)
y_val.to_csv(f'{OUTPUT_DIR}/y_val_prep.csv', index=False)
y_test.to_csv(f'{OUTPUT_DIR}/y_test_prep.csv', index=False)

print(f"✅ Данните са успешно записани в {OUTPUT_DIR}")

## Заключение

В този ноутбук успешно:
1.  Идентифицирахме и "поправихме" (симулативно) липсващи данни и отклонения.
2.  Създадохме **3 нови признака** (`Burnout_Index`, `Salary_Per_Year`, `Efficiency`), които добавят стойност към модела.
3.  Трансформирахме всички данни в числен вид (OneHot + Scaling) чрез **Pipeline**.
4.  Гарантирахме, че няма изтичане на информация чрез стриктно разделяне на Train/Val/Test преди fit().